# A fresh start with Qwen3.8

**Model upgrade · Verify the new pair before rebuilding the experiment**

We are moving to a newer teacher and a compatible small student. This notebook
checks their tokenizers, loads their text components, and follows one student
response through frozen-teacher scoring. It does not reuse the old model weights.

| Role | New pinned model |
| :--- | :--- |
| Student | `Qwen/Qwen3.5-0.8B-Base` |
| Teacher | `Qwen/Qwen3.8-27B` |
| Text architecture | `Qwen3_5ForCausalLM` |
| Stopping IDs | `248044` (`endoftext`); `248046` (`im_end`) |
| Starting mode | CPU inspection, with no model weights |
| GPU milestone | One sampled response and correctly aligned teacher scores |

![The model-upgrade checks](assets/05_qwen38_restart/upgrade_workflow.png)

*First establish compatibility. Then measure the new student, build a new
supervised checkpoint, and run a new guarded distillation experiment.*

**Research status:** a newer model is a new experimental choice. It is not yet an
established fix for the earlier training deterioration. Notebooks 01-04 and their
recorded results describe the original Qwen3 models.
**Read in order** · [Goal](#qwen38-goal) · [Runtime](#qwen38-runtime) ·
[Token meanings](#qwen38-tokens) · [Generation](#qwen38-generate) ·
[Alignment](#qwen38-alignment) · [Evidence](#qwen38-evidence)


<a id="qwen38-goal"></a>
## 1. Why this needs a fresh start

Changing a model changes its learned weights, architecture, tokenizer, and sometimes
its ending convention. The old 03 checkpoint cannot become Qwen3.5 by changing its
name. We keep it for reproducing the old experiment.

The previous comparison taught us something useful: grouping ending tokens reduced
stopping failures, but the student still lost answer accuracy. The locally audited
validation counts were 33/64 before distillation, 1/64 after the raw-token arm, and
23/64 after the grouped-stop arm. These results remain evidence for that old pair.
They are not scores for the models introduced here.

Qwen's official Qwen3.8 release has no comparably small student. The selected
Qwen3.5 base student uses the same text architecture family as this Qwen3.8 teacher.
We verify token meanings explicitly rather than assuming compatibility from names.

![The student and teacher text architectures](assets/05_qwen38_restart/model_family.png)

*Layer counts come from the pinned model configurations. This drawing describes
architecture, without making a performance claim.*

### What the model names change

The student is the smaller model whose weights we will eventually update. The
teacher supplies a larger model's probabilities. Both use the `Qwen3_5` text
architecture implementation, even though their release names differ.

These models mix linear attention and full attention. Linear attention maintains
a compact state of earlier text. Full attention attends to stored information
from earlier token positions. That hybrid cache differs from the older Qwen3
cache, which is why we test generation as well as a single forward pass.

**Section checkpoint.** We have selected the two model identities. A new baseline
and supervised checkpoint will establish what this student can do before distillation.


<a id="qwen38-runtime"></a>
## 2. Prepare the new environment

Use `requirements-qwen38.txt` in a separate user environment. The old run used
Transformers 4.57.6; this pair is checked with 5.3.0. Keep the original environment
available for reproducing the old experiment. See the [run guide](assets/05_qwen38_restart/RUN.md).

`MODE = "inspect"` loads only configurations and tokenizers. After that succeeds,
request GPUs, select that Jupyter kernel, change the mode to `"gpu_check"`, and run
from the top. The GPU check uses one almost-empty 80GB GPU or two almost-empty
40GB-or-larger GPUs. The new real-model check has not been run locally.

The teacher uses hybrid linear and full attention. Without optional fast kernels,
Transformers falls back to slower PyTorch operations. We start with this supported
reference path, without compiling extensions or changing the cluster system.

In [ ]:
import importlib.metadata
import os
import sys
from pathlib import Path

# Inspection downloads small tokenizer/configuration files, not the model weights.
# Use the environment override only for separate QA kernels or explicit batch runs.
MODE = os.environ.get("OPD_QWEN38_MODE", "inspect")
assert MODE in {"inspect", "gpu_check"}
PROJECT_ROOT = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents, Path.home() / "on-policy-distillation"]
        if (p / "src/qwen38/model_upgrade.py").is_file()
    ),
    None,
)
assert PROJECT_ROOT is not None, "Sync the complete updated project to the remote server."
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Set the scratch cache before importing Hugging Face libraries on Amarel.
# Reports and trained checkpoints belong in persistent project storage.
scratch = Path("/scratch") / os.environ.get("USER", "")
if os.environ.get("SLURM_JOB_ID") and scratch.is_dir():
    os.environ.setdefault("HF_HOME", str(scratch / "huggingface"))
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
assert importlib.metadata.version("transformers") == "5.3.0", (
    "Select the new environment with requirements-qwen38.txt, then restart the kernel."
)
print("Mode:", MODE)
print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

from src.qwen38.model_upgrade import (
    inspect_upgrade,
    load_upgrade_models,
    load_upgrade_profile,
    score_upgrade_trajectory,
)
from src.utils import verified_zip, write_inventory, write_json

# Immutable Hub revisions identify the exact starting weights and tokenizers.
# Old 03 checkpoint paths are intentionally absent from this configuration.
profile = load_upgrade_profile(PROJECT_ROOT)
display(
    pd.DataFrame(
        [
            {
                "role": role,
                "model": profile[f"{role}_model_id"],
                "revision": profile[f"{role}_revision"],
            }
            for role in ("student", "teacher")
        ]
    )
)

<a id="qwen38-tokens"></a>
## 3. Verify that token IDs mean the same thing

A token ID is a numbered piece of text. To compare complete next-token
distributions, column 100 must mean the same token for both models.

The raw student `tokenizer.json` omits 11 tokens present in the teacher's file.
The student's official `tokenizer_config.json` supplies them. The supported
`AutoTokenizer` loader reads both files and resolves the same **248,077 token
mappings** for both models. We neither renumber tokens nor resize the embeddings.

Both language-model output heads contain **248,320 columns**. Extra columns are
padding in the model architecture; equal head sizes alone do not establish token
compatibility. The check below compares the resolved mappings, BPE rules, encoding,
decoding, and both ending IDs on representative text and 1,000 numeric examples.

![How the tokenizer files resolve to identical token mappings](assets/05_qwen38_restart/tokenizer_resolution.png)

*The 11 additional definitions are supplied by the official student configuration.
Loading both files resolves the same ID-to-text mapping.*

### Tokenizer entries and output columns

For example, a token ID can represent a leading space plus a word. A logit is the
model's score for one possible next ID. The tokenizer tells us what that ID means.
An output head reserves 248,320 columns, while the resolved tokenizer defines
248,077 entries. The extra 243 rows are architectural padding.

We compare the whole output head when calculating KL. We also check that sampled
IDs have tokenizer entries, so an undefined row cannot silently become an empty
piece of decoded response text. The probability assigned to padded rows is logged.


In [ ]:
# QA can point to already-downloaded files. Normal notebook use fetches the exact
# pinned tokenizer/configuration files from the Hub, without loading weights.
local_sources = os.environ.get("OPD_QWEN38_LOCAL_SOURCES")
sources = Path(local_sources) if local_sources else None
tokenizers, model_configs, compatibility = inspect_upgrade(profile, sources=sources)
print(json.dumps(compatibility, indent=2))
assert compatibility["full_tokenizers_identical"]
assert compatibility["vocabulary_mode"] == "full_head"

In [ ]:
# This example uses the ACTUAL pinned tokenizers, while loading no model weights.
# Each row makes one shared token meaning visible to the reader.
example_text = "24 boxes contain 15 pencils."
student_example_ids = tokenizers["student"].encode(example_text, add_special_tokens=False)
teacher_example_ids = tokenizers["teacher"].encode(example_text, add_special_tokens=False)
assert student_example_ids == teacher_example_ids
print("Text:", example_text)
display(
    pd.DataFrame(
        {
            "position": range(len(student_example_ids)),
            "student_token_id": student_example_ids,
            "teacher_token_id": teacher_example_ids,
            "decoded_token": [
                repr(tokenizers["student"].decode([token_id])) for token_id in student_example_ids
            ],
        }
    )
)

<a id="qwen38-generate"></a>
## 4. Let the new student write one response

For the GPU check, the student starts from its original base weights. It has not
received the new supervised cold start yet, so a weak answer is possible.

We sample with temperature 1, no top-k/top-p truncation, and seed 42. We do not
mask vocabulary columns or change the sampled IDs. If the student emits an unused
output-head row with no tokenizer entry, the check fails explicitly.

The teacher's frozen BF16 layers can span two GPUs. The FP32 student stays on logical
GPU 0. Reserved memory is a planning estimate, not evidence that the full training
run will fit. Never call `teacher.to(...)` after sharding the teacher.

![Illustrative teacher placement across two allocated GPUs](assets/05_qwen38_restart/gpu_layout.png)

*GPU memory belongs to each card separately. The actual device map and measured
peak memory depend on the allocation and are recorded during qualification.*

### Why two numeric precisions appear

The frozen teacher uses BF16 to reduce weight memory. The student weights remain
FP32 for the future training workflow. During scoring, CUDA autocast selects BF16
for supported operations. We convert score calculations to FP32 before softmax
because the exponential and logarithm operations benefit from greater precision.

Sampling draws one possible answer from the student's distribution. Seed 42 makes
that draw repeatable on the same hardware/software setup. The generated response
IDs are preserved so both models can score the same text prefix at every position.


### GPU qualification before long training

GPU mode first preserves one real student response and teacher scores. It then
uses clearly labeled synthetic IDs to test a 256-token response backward pass,
one disposable update, a 1,024-position cached student generation, a 1,056-position
teacher forward, and checkpoint reload. The extra teacher positions cover chat-prefix
overhead. These probes measure execution and memory. Their temporary weights are
discarded. Actual SFT and KD runs begin later from the pinned pretrained weights.

The forced-length memory probe uses its own generation settings. Research trajectories
continue to use unfiltered sampling at temperature 1.


In [ ]:
# This branch loads REAL pretrained weights only when explicitly selected.
# The helper returns original prompt/response IDs and each model's causal offset.
real_model_result = None
qualification_result = None
if MODE == "gpu_check":
    import torch

    assert torch.cuda.is_available(), "Connect to an allocated CUDA kernel."
    print("Visible GPUs:", torch.cuda.device_count())
    qualification_directory = (
        PROJECT_ROOT
        / "results"
        / ("qwen38_qualification_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    )
    qualification_directory.mkdir(parents=True, exist_ok=False)
    student, teacher = load_upgrade_models(profile, model_configs)
    try:
        prompt = (
            "A store has 24 boxes. Each box contains 15 pencils. The store sells 87 pencils. "
            "How many pencils remain? Explain your reasoning step by step. "
            "End with one final line: #### <number>."
        )
        real_model_result = score_upgrade_trajectory(
            student, teacher, tokenizers, profile, prompt, max_new_tokens=64
        )
        print("Student response:\n", real_model_result["response_text"])
        print("Response token IDs:", real_model_result["response_token_ids"])
        print("Status:", real_model_result["status"])
        # Persist real scoring before the longer synthetic memory/update probes.
        write_json(qualification_directory / "scoring.json", real_model_result)
        from src.qwen38.qualification import qualify_gpu

        qualification_result = qualify_gpu(student, teacher, tokenizers, profile, PROJECT_ROOT)
        write_json(qualification_directory / "qualification.json", qualification_result)
        print("Qualification:", qualification_result["status"])
    except BaseException as error:
        write_json(
            qualification_directory / "failure.json",
            {"error_type": type(error).__name__, "error": str(error)},
        )
        raise
    finally:
        # Clear notebook references so another attempt does not retain a teacher.
        del student, teacher
        from src.opd import release_cuda_memory

        release_cuda_memory()
else:
    print("INSPECT: real student generation and teacher scoring remain pending on CUDA.")

<a id="qwen38-alignment"></a>
## 5. Read the alignment and probability checks

If a prompt has 73 tokens, its response starts at input position 73. The logits at
position **72** predict that first response token. The teacher's chat-formatted
prompt has a different length, so its predicting position is calculated separately.
Both models then read **exactly the same generated response token IDs**.

The aligned logits have shape `[1, response_length, 248320]`: one response, one
prediction for each generated token, and one score for every output-head column.
Full-head forward KL uses FP32 probability arithmetic and groups the two ending
IDs into one STOP event. No vocabulary projection is applied.

Here the teacher temperature is **1**, for an ordinary-probability diagnostic.
This is not a selected training hyperparameter. No backward pass or optimizer runs.
The check also records teacher probability mass on padded rows so it stays visible.

![A worked example of the two causal prediction offsets](assets/05_qwen38_restart/causal_alignment.png)

*The illustrated prompt lengths are 3 and 5. Actual lengths are printed from the
real trajectory. Both response slices contain the same three illustrated IDs.*

### Follow the first response token

Suppose the student prompt has three tokens. Its first response token has input
index 3, and logits at index 2 predict it. If the teacher prompt has five tokens,
that same response token has input index 5 and uses teacher logits at index 4.
The extra teacher prompt tokens come from chat formatting. We slice each sequence
at its own boundary and compare the resulting response positions.


In [ ]:
import torch

# Illustrative IDs demonstrate positions. They are separate from a real trajectory.
# Each input tensor has shape [batch_size=1, prompt_tokens + response_tokens].
example_response_ids = [42, 43, profile["eos_token_ids"][1]]
example_student_sequence = torch.tensor([[11, 12, 13] + example_response_ids])
example_teacher_sequence = torch.tensor([[7, 8, 11, 12, 13] + example_response_ids])
example_student_prompt_length = 3
example_teacher_prompt_length = 5
assert torch.equal(
    example_student_sequence[:, example_student_prompt_length:],
    example_teacher_sequence[:, example_teacher_prompt_length:],
)
display(
    pd.DataFrame(
        [
            {
                "model": "student",
                "prompt_length": 3,
                "first_response_input_position": 3,
                "predicting_logits_position": 2,
            },
            {
                "model": "teacher",
                "prompt_length": 5,
                "first_response_input_position": 5,
                "predicting_logits_position": 4,
            },
        ]
    )
)

In [ ]:
# Each row corresponds to one token the student actually sampled.
# repr() exposes leading spaces and newline characters in decoded tokens.
if real_model_result is not None:
    response_token_ids = real_model_result["response_token_ids"]
    display(
        pd.DataFrame(
            {
                "response_position": range(len(response_token_ids)),
                "token_id": response_token_ids,
                "decoded_token": [
                    repr(tokenizers["student"].decode([token_id]))
                    for token_id in response_token_ids
                ],
                "student_log_probability_T1": real_model_result["student_log_probs_T1"],
                "teacher_log_probability_T1": real_model_result["teacher_log_probs_T1"],
            }
        ).head(30)
    )
    for key in (
        "student_prompt_length",
        "teacher_prompt_length",
        "response_length",
        "student_predicting_first_response_position",
        "teacher_predicting_first_response_position",
        "aligned_logits_shape",
        "ended_with_eos",
        "hit_token_limit",
        "full_head_semantic_stop_kl_T1",
        "teacher_padded_row_probability_mass_max_T1",
    ):
        print(f"{key}:", real_model_result[key])
    assert real_model_result["weights_updated"] is False
else:
    print("PENDING: no generated-token table or real-model KL was measured in inspect mode.")

### 5.1 Why ending tokens are grouped

The earlier experiment showed that a student and teacher can agree that an answer
should end while preferring different ending-token IDs. A token-level loss then
penalizes the representation of ending. Grouping combines their ending probabilities
into one STOP event before measuring disagreement.

$$
p(\mathrm{STOP}) = p(248044) + p(248046).
$$

![Illustrative probability distributions before and after grouping EOS](assets/05_qwen38_restart/eos_probabilities.png)

*These three-action distributions are invented teaching examples. The real score
calculation retains all output columns and uses the model's actual probabilities.*

The next cell makes the two distributions agree on total STOP probability. It
shows why grouping removes this particular penalty. The earlier grouped pilot
still lost accuracy, so answer quality must be checked alongside stopping behavior.

In [ ]:
from src.opd_loss import forward_kl_loss

# Three illustrative columns represent continue, endoftext, and im_end.
# Shape [1, 1, 3] means one batch item, one prediction, and three possible choices.
example_student_logits = torch.tensor([[[0.20, 0.79, 0.01]]]).log()
example_teacher_logits = torch.tensor([[[0.20, 0.01, 0.79]]]).log()
example_raw_kl = forward_kl_loss(
    example_student_logits,
    example_teacher_logits,
    teacher_temperature=1.0,
    recompute=False,
)
example_grouped_kl = forward_kl_loss(
    example_student_logits,
    example_teacher_logits,
    teacher_temperature=1.0,
    eos_token_ids=(1, 2),
    recompute=False,
)
print("Illustrative raw-token KL:", float(example_raw_kl))
print("Illustrative grouped-STOP KL:", float(example_grouped_kl))
assert abs(float(example_grouped_kl)) < 1e-6

<a id="qwen38-evidence"></a>
## 6. Save the readiness evidence

The report records the exact model revisions, configuration, tokenizer checks,
execution mode, and any actual GPU result. An inspection report does not establish
GPU memory usage, trained quality, or successful optimization. The ZIP is small;
it contains no model weights or optimizer state.

### How future training protects progress

![Checkpoint and validation decisions for a future guarded pilot](assets/05_qwen38_restart/checkpoint_safety.png)

*This is the training workflow being prepared for the later pilot. The current
readiness check performs generation and scoring.*

At a training boundary, the optimizer state and weights are saved before validation.
The run then checks reference-token loss, strict numeric matches, and the recent
frequency of token-limit stops. A quality pause preserves both the degraded state
for diagnosis and the last acceptable checkpoint for recovery.

The chosen engineering limits pause when reference loss rises over 10%, strict
accuracy drops over five percentage points, or more than eight of the latest 32
sampled responses are capped. These limits apply relative to that run's starting
checkpoint. They request a review when learning deteriorates.


In [ ]:
from dataclasses import asdict

from src.training_safety import TrainingSafetyConfig, assess_training_safety

# This synthetic example demonstrates guard behavior without training a model.
# Starting NLL is 1.0 and starting strict accuracy is 80 percent.
example_safety_policy = TrainingSafetyConfig()
example_degraded_history = [
    {
        "step": 8,
        "validation_reference_nll": 1.20,
        "generated_validation": {"strict_accuracy": 0.65},
        "response_limit_flags": [True] * 9 + [False] * 23,
    }
]
example_safety_result = assess_training_safety(
    example_degraded_history,
    1.0,
    0.80,
    example_safety_policy,
)
print("Configured limits:", asdict(example_safety_policy))
display(pd.DataFrame(example_safety_result["reasons"]))
assert example_safety_result["pause"]
print("The example requests a pause before another optimizer update.")

In [ ]:
# Use a fresh directory so a later run cannot overwrite an earlier result.
run_directory = (
    PROJECT_ROOT
    / "results"
    / ("qwen38_readiness_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
)
run_directory.mkdir(parents=True, exist_ok=False)
report = {
    "mode": MODE,
    "profile": profile,
    "tokenizer_check": compatibility,
    "real_model_scoring": real_model_result,
    "training_performed": qualification_result is not None,
    "training_scope": "one_disposable_synthetic_update" if qualification_result else None,
    "experiment_training_performed": False,
    "gpu_qualification": qualification_result,
    "old_checkpoint_reused": False,
}
write_json(run_directory / "readiness.json", report)
write_inventory(run_directory)
report_archive = verified_zip(run_directory, run_directory.with_suffix(".zip"))
print("Readiness report:", report_archive)

## What we achieved, in simple terms

We selected a newer teacher and a compatible small student, pinned their versions,
and verified that their loaded tokenizers assign the same meanings to the same IDs.
We kept the original experiment and its failed pilots as historical evidence.

In **inspect mode**, only the configuration and tokenizer checks run. In
**gpu_check mode**, the new student writes one response and the frozen teacher
scores those exact tokens. Nothing in this notebook changes either model's weights.

### What comes next

1. Run the real-model GPU check and inspect its response, shapes, finite values,
   ending behavior, and memory use.
2. Measure a fresh baseline and train a new supervised cold start for this student.
   Use new output folders and record the new model/tokenizer identities. The old
   Qwen3 03 archive cannot be reused as the new student's checkpoint.
3. Recheck teacher scoring on that new checkpoint, then run a guarded distillation
   pilot. Answer quality and stopping behavior both need to pass before extending it.

The older 02/03/04 scores are not measurements of this new pair. A newer model may
help, but it cannot by itself establish that our learning settings are appropriate.

### Sources

- [Official Qwen3.8 teacher](https://huggingface.co/Qwen/Qwen3.8-27B)
- [Official Qwen3.5 base student](https://huggingface.co/Qwen/Qwen3.5-0.8B-Base)
- [Transformers Qwen3.5 text-only model documentation](https://huggingface.co/docs/transformers/v5.3.0/en/model_doc/qwen3_5)
- [Generalized Knowledge Distillation](https://arxiv.org/abs/2306.13649)

### Read the evidence at the right stage

A successful tokenizer check establishes that the IDs have shared meanings. A
successful scoring check establishes that one trajectory can be aligned and scored.
A successful supervised run establishes a new checkpoint and its measured behavior.
Finally, a guarded pilot measures whether teacher feedback improves that checkpoint.

The latest user-run Amarel inspection confirms tokenizer compatibility for this
pair. Its readiness report records `real_model_scoring = null`, so the real-model
GPU milestone is still pending in the evidence available here.

The collected older-model comparison now has verified trajectories and checker
replay. Its strict accuracy first declined at update 8 in both arms. That finding
motivates checking answer quality during the future pilot for this new pair.
